In [6]:
import pandas as pd
from google.colab import files


uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)


TARGET_COLUMN = "PlayTennis"

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found."
    )

# Remove missing values
df = df.dropna().reset_index(drop=True)

# Features
X = df.drop(columns=[TARGET_COLUMN])

# Target
y = df[TARGET_COLUMN]

print("\nFeatures:")
print(list(X.columns))

print("\nTarget:")
print(TARGET_COLUMN)

print("\nClass distribution:")
print(y.value_counts())

examples = X.values.tolist()
labels = y.tolist()

n_attributes = len(X.columns)

print("\n" + "=" * 70)
print("TRAINING DATA")
print("=" * 70)

for i, (example, label) in enumerate(
        zip(examples, labels), start=1):

    print(f"{i:2d}. {example} -> {label}")


def fulfills(example, hypothesis):

    for h, e in zip(hypothesis, example):

        # null covers nothing
        if h == "null":
            return False

        # ? matches every value
        if h == "?":
            continue

        # Specific value must match
        if h != e:
            return False

    return True

def more_general(h1, h2):

    for x, y in zip(h1, h2):

        # ? is maximally general
        if x == "?":
            continue

        # null is maximally specific
        if x == "null":

            if y != "null":
                return False

            continue

        # Specific value must match
        if x != y and y != "null":
            return False

    return True


def min_generalizations(h, example):

    new_h = list(h)

    for i in range(len(h)):

        # null -> example value
        if h[i] == "null":
            new_h[i] = example[i]

        # Different value -> ?
        elif h[i] != example[i]:
            new_h[i] = "?"

    return [tuple(new_h)]



def min_specializations(h, domains, example):

    results = []

    for i in range(len(h)):

        # Only ? can be specialized
        if h[i] == "?":

            for value in domains[i]:

                # Don't use the value of the
                # negative example
                if value != example[i]:

                    new_h = list(h)
                    new_h[i] = value

                    results.append(tuple(new_h))

    return results


def remove_duplicates(hypotheses):

    return list(dict.fromkeys(hypotheses))


def prune_S(S):

    result = []

    for s in S:

        is_more_general = False

        for other in S:

            if s != other and more_general(s, other):

                is_more_general = True
                break

        if not is_more_general:
            result.append(s)

    return remove_duplicates(result)


def prune_G(G):

    result = []

    for g in G:

        is_more_specific = False

        for other in G:

            if g != other and more_general(other, g):

                is_more_specific = True
                break

        if not is_more_specific:
            result.append(g)

    return remove_duplicates(result)


def candidate_elimination(examples, labels):

    n_attributes = len(examples[0])

    domains = []

    for i in range(n_attributes):

        values = []

        for example in examples:

            if example[i] not in values:
                values.append(example[i])

        domains.append(values)


    S = [
        tuple(["null"] * n_attributes)
    ]

    G = [
        tuple(["?"] * n_attributes)
    ]


    for example, label in zip(examples, labels):

        example = tuple(example)

        positive = (
            str(label).strip().lower()
            in ["yes", "true", "1"]
        )


        if positive:

            # Remove G hypotheses that don't cover
            # the positive example
            G = [
                g for g in G
                if fulfills(example, g)
            ]

            new_S = []

            for s in S:

                # If S already covers example
                if fulfills(example, s):

                    new_S.append(s)

                else:

                    # Generalize S
                    generalizations = min_generalizations(
                        s,
                        example
                    )

                    for h in generalizations:

                        # Keep h if some G is
                        # more general than h
                        if any(
                            more_general(g, h)
                            for g in G
                        ):

                            new_S.append(h)

            S = prune_S(new_S)


        else:

            # Remove S hypotheses that cover
            # the negative example
            S = [
                s for s in S
                if not fulfills(example, s)
            ]

            new_G = []

            for g in G:

                # If G doesn't cover negative example
                if not fulfills(example, g):

                    new_G.append(g)

                else:

                    # Specialize G
                    specializations = min_specializations(
                        g,
                        domains,
                        example
                    )

                    for h in specializations:

                        # Keep h if it is still
                        # more general than some S
                        if any(
                            more_general(h, s)
                            for s in S
                        ):

                            new_G.append(h)

            G = prune_G(new_G)


    return S, G

final_S, final_G = candidate_elimination(
    examples,
    labels
)

print("\n")
print("=" * 70)
print("FINAL VERSION SPACE")
print("=" * 70)


print("\nMost Specific Boundary (S):")

if final_S:

    for s in final_S:
        print(s)

else:

    print("EMPTY")


print("\nMost General Boundary (G):")

if final_G:

    for g in final_G:
        print(g)

else:

    print("EMPTY")


Saving playtennis.csv to playtennis (3).csv

Features:
['Outlook', 'Temperature', 'Humidity', 'Wind']

Target:
PlayTennis

Class distribution:
PlayTennis
Yes    105
No      76
Name: count, dtype: int64

TRAINING DATA
 1. ['Sunny', 'Hot', 'High', 'Weak'] -> No
 2. ['Overcast', 'Mild', 'Normal', 'Strong'] -> No
 3. ['Rain', 'Mild', 'High', 'Strong'] -> No
 4. ['Rain', 'Hot', 'Normal', 'Weak'] -> Yes
 5. ['Sunny', 'Mild', 'High', 'Weak'] -> No
 6. ['Overcast', 'Mild', 'Normal', 'Strong'] -> Yes
 7. ['Sunny', 'Hot', 'High', 'Strong'] -> No
 8. ['Overcast', 'Mild', 'Normal', 'Weak'] -> Yes
 9. ['Overcast', 'Hot', 'Normal', 'Weak'] -> Yes
10. ['Rain', 'Cool', 'High', 'Weak'] -> No
11. ['Overcast', 'Hot', 'Normal', 'Weak'] -> Yes
12. ['Overcast', 'Hot', 'Normal', 'Strong'] -> Yes
13. ['Sunny', 'Mild', 'Normal', 'Weak'] -> Yes
14. ['Sunny', 'Cool', 'Normal', 'Strong'] -> Yes
15. ['Rain', 'Mild', 'High', 'Strong'] -> No
16. ['Overcast', 'Hot', 'High', 'Weak'] -> Yes
17. ['Sunny', 'Mild', 'Norma